# Preparation: imports and data load

### imports

In [ ]:
# standard
import numpy as np
import matplotlib.pyplot as plt
import importlib
import os
import h5py
import json
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import time
import math
import gc


# custom

import axolotl_utils_ram
importlib.reload(axolotl_utils_ram)

from axolotl_utils_ram import extract_snippets_fast_ram, compute_and_plot_sta


import collision_utils
importlib.reload(collision_utils)

from collision_utils import (
        select_template_channels,
        main_channel_and_neg_peak,
        compute_harm_map_noamp,
        plot_harm_heatmap,
        scan_continuous_harm_regions,
        median_ei_adaptive,
)

import joint_utils
importlib.reload(joint_utils)

from joint_utils import (
    split_first_good_channel_and_visualize,
    split_first_good_channel_subset_and_harm,
    cosine_two_eis,
    classify_two_cells_vs_ab_shard,
    plot_grouped_harm_maps_two_eis,
    check_bimodality_and_plot,
    recenter_ei_to_ref_trough,
    compute_ei_from_indices,
    assess_ei_drift,
    kmeans_split_diagnostics,
    detect_negative_peaks_on_channel,
    select_top_by_amplitude,
    extract_and_build_ei,
    harm_map_diagnostic,
    amplitude_gate_pruner,
    two_cells_or_ab_shard_pruner,
    harm_compliance_pruner,
    bimodality_split_pruner,
    make_mask_entry,
    mask_trace_with_template_bank,
    save_mask_bank_pickle,
    load_mask_bank_pickle,
    find_ks_ax_matches,
    best_crossmatches, 
    cosine_two_eis_asym
    
)

import lighthouse_utils
importlib.reload(lighthouse_utils)

from lighthouse_utils import (
    find_valley_and_times,
    build_left_low_high_eis,
    filter_valley_spikes_by_batches,
    finalize_ei_with_cap,
    compute_prev_metrics, 
    plot_amp_vs_prev_metrics,
    plot_amp_prev_scatter_and_bar,
    lighthouse_morpho_pc_gmm
)


import plot_ei_waveforms
importlib.reload(plot_ei_waveforms)
import plot_ei_waveforms as pew

import collision_utils
importlib.reload(collision_utils)


import compute_sta_from_spikes
importlib.reload(compute_sta_from_spikes)


import joint_utils
importlib.reload(joint_utils)

from joint_utils import (
    cosine_two_eis
)

### load raw data and subtract baselines

In [ ]:
# --- Path and recording setup ---
dat_path = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004.dat"
n_channels = 512
dtype = np.int16

# Sampling rate (your usual)
fs = 20_000

# What chunk to load
start_minutes   = 0
minutes_to_load = 30

# --- Get total number of samples in file ---
file_size_bytes = os.path.getsize(dat_path)
total_samples_in_file = file_size_bytes // (np.dtype(dtype).itemsize * n_channels)

# --- Convert desired time window -> samples ---
start_sample = int(start_minutes * 60 * fs)
n_samples    = int(minutes_to_load * 60 * fs)

# Clamp to file bounds
if start_sample >= total_samples_in_file:
    raise ValueError(f"start_sample={start_sample} beyond file length {total_samples_in_file}")

n_samples = min(n_samples, total_samples_in_file - start_sample)

# --- Read only that chunk ---
offset_bytes = start_sample * n_channels * np.dtype(dtype).itemsize
count_vals   = n_samples * n_channels

with open(dat_path, "rb") as f:
    f.seek(offset_bytes, os.SEEK_SET)
    raw_data = np.fromfile(f, dtype=dtype, count=count_vals)

raw_data = raw_data.reshape((n_samples, n_channels))  # [T, C]
print(f"Loaded raw_data {raw_data.shape} = {n_samples/fs/60:.1f} minutes")



# ONLY EI POSITIONS
h5_in_path = '/Volumes/Lab/Users/alexth/axolotl/201703151_kilosort_data001_spike_times.h5'  # from MATLAB export, to get EI positions

with h5py.File(h5_in_path, 'r') as f:
    # Load electrode positions
    ei_positions = f['/ei_positions'][:].T  # shape becomes [512 x 2]



baseline_path = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_baseline_derivative_20k_30min.json"

segment_len = 20_000
if os.path.exists(baseline_path):
    print(f"Loading baselines")
    with open(baseline_path, 'r') as f:
        data = json.load(f)
    baselines = np.array(data['baselines'], dtype=np.float32)
else:
    print(f"Computing baselines")
    baselines = axolotl_utils_ram.compute_baselines_int16_deriv_robust(raw_data, segment_len=segment_len, diff_thresh=10, trim_fraction=0.15) # shape (512, 360)

    with open(baseline_path, 'w') as f:
        json.dump({
            'baselines': baselines.tolist(),
        }, f)

print("subtracting baselines")

axolotl_utils_ram.subtract_segment_baselines_int16(raw_data=raw_data,
                                     baselines_f32=baselines,
                                     segment_len=segment_len) 




### ALTERNATIVE: load MODIFIED raw data

In [ ]:
OUTDIR = "/Volumes/Lab/Users/alexth/axolotl/201711290"
ROUND_LABEL = "round5"
base = f"data004_lighthouse_residual_{ROUND_LABEL}"

dat_path  = os.path.join(OUTDIR, base + ".dat")
json_path = os.path.join(OUTDIR, base + ".json")

assert os.path.exists(dat_path), f"Missing DAT: {dat_path}"
assert os.path.exists(json_path), f"Missing JSON: {json_path}"

with open(json_path, "r") as f:
    meta = json.load(f)

# Get shape + dtype from metadata
total_samples, n_channels = map(int, meta["shape"])
dtype = np.int16  # you wrote int16

# (Optional but good) sanity check file size
bytes_actual = os.path.getsize(dat_path)
bytes_expected = total_samples * n_channels * np.dtype(dtype).itemsize
assert bytes_actual == bytes_expected, (
    f"Size mismatch: expected {bytes_expected} bytes for ({total_samples},{n_channels}) int16, got {bytes_actual}"
)

# ---- ONE RAM ALLOCATION: read the entire file into memory ----
raw_data = np.fromfile(dat_path, dtype=dtype).reshape(total_samples, n_channels)

# Make sure it’s writable (it will be by default)
assert raw_data.flags.writeable, "raw_data is not writeable (unexpected)."

# Restore sample rate if you want it in globals again
sample_rate_hz = float(meta.get("sample_rate_hz", 20_000))

print(f"✅ raw_data loaded into RAM: shape={raw_data.shape}, dtype={raw_data.dtype}, writeable={raw_data.flags.writeable}")
print(f"   sample_rate_hz={sample_rate_hz}")


# supporting stuff
fs = 20_000 # Sampling rate

# ONLY EI POSITIONS
h5_in_path = '/Volumes/Lab/Users/alexth/axolotl/201703151_kilosort_data001_spike_times.h5'  # from MATLAB export, to get EI positions

with h5py.File(h5_in_path, 'r') as f:
    # Load electrode positions
    ei_positions = f['/ei_positions'][:].T  # shape becomes [512 x 2]

### copy original for easy recovery

In [ ]:

raw_orig = raw_data                      # keep a name for “do not touch”
raw_orig.setflags(write=False)           # any accidental write will error

raw_sub = raw_orig.copy()                # your working residual
raw_sub.setflags(write=True)

print("raw_orig dtype/shape:", raw_orig.dtype, raw_orig.shape, "GB:", raw_orig.nbytes/1e9)
print("raw_sub  dtype/shape:", raw_sub.dtype,  raw_sub.shape,  "GB:", raw_sub.nbytes/1e9)
print("shares_memory:", np.shares_memory(raw_orig, raw_sub))

# Rename to the only two names we will use
raw_mod = raw_sub        # working copy you will subtract into
# raw_orig already exists and is the frozen original

# Kill the confusing aliases (optional but recommended)
del raw_sub
del raw_data             # raw_data was just another name for raw_orig

# Safety flags
raw_orig.setflags(write=False)  # original is read-only
raw_mod.setflags(write=True)    # working copy is writable

print("raw_orig is frozen, raw_mod is mutable.")
print("shares_memory:", np.shares_memory(raw_orig, raw_mod))

### restore from original

In [ ]:
raw_mod[:] = raw_orig

### load triggers and onsets

In [ ]:
path = '/Volumes/Lab/Users/alexth/axolotl/201711290/data004_triggers.h5'
with h5py.File(path, 'r') as h5:
    triggers_sec = h5['/triggers'][...]         # shape (11500, 1) or (11500,)
triggers_sec = np.ravel(triggers_sec).astype(np.float64)  # 1D float64

triggers_sec = triggers_sec[triggers_sec < 30*60]   # keep only triggers in first 30 minutes

len(triggers_sec)  # should be 11500 for full

def read_onsets(path):
    with h5py.File(path, "r") as h5:
        on  = np.ravel(h5["/onsets_sec"][...]).astype(np.float64)
        dur = np.ravel(h5["/dur_sec"][...]).astype(np.float64)
        fr  = np.ravel(h5["/frame_idx"][...]).astype(np.int64)   # 0-based
        blk = np.ravel(h5["/block_idx"][...]).astype(np.int64)   # 0-based
        tri = np.ravel(h5["/trial_in_block"][...]).astype(np.int64)
    return on, dur, fr, blk, tri

# ---- UNIQUE ----
path_u = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_onset_unique.h5"
on_u, dur_u, fr_u, blk_u, tri_u = read_onsets(path_u)

# optional truncate to first 30 minutes (keeps alignment across arrays)
keep = np.isfinite(on_u) & (on_u < 30*60)
on_u30, dur_u30, fr_u30 = on_u[keep], dur_u[keep], fr_u[keep]
len(on_u), len(on_u30)

# ---- REPEATS ----
path_r = "/Volumes/Lab/Users/alexth/axolotl/201711290/data004_onset_repeat.h5"
on_r, dur_r, fr_r, blk_r, tri_r = read_onsets(path_r)

keep = np.isfinite(on_r) & (on_r < 30*60)
on_r30, dur_r30, fr_r30 = on_r[keep], dur_r[keep], fr_r[keep]
len(on_r), len(on_r30)

# ---- spike-window extraction helper (per onset) ----
# spikes_sec should be a sorted 1D float64 array for a cell
def spikes_in_trial(spikes_sec, onset_sec, dur_sec=0.5):
    # Use dur_sec from file: it already applies min(0.5, next_onset-onset)
    a = onset_sec
    b = onset_sec + dur_sec
    i0 = np.searchsorted(spikes_sec, a, side="left")
    i1 = np.searchsorted(spikes_sec, b, side="left")
    return spikes_sec[i0:i1] - onset_sec  # relative times in [0,dur)

# Example usage:
# rel = spikes_in_trial(spikes_sec, on_r[123], dur_r[123])  # repeat presentation 123

# Load EIs from KS (white noise), recenter, cut to 121

In [ ]:
with h5py.File('/Volumes/Lab/Users/alexth/axolotl/201711290/data999_full_ei.h5','r') as h5:
    EIs = h5['/EIs'][...]   # currently (N, T, C) = (1217, 201, 512)

# Make it (C, T, N) = (512, 201, 1217)
EIs = np.transpose(EIs, (2, 1, 0))

# Python list of [C,T] arrays (one per unit):
ei_list = [EIs[:, :, i] for i in range(EIs.shape[2])]

# EIs: shape (C, T, N) == (512, 201, num_units)
C, T, N = EIs.shape
ci = 40          # target peak index in the recentered window (0-based)
L  = 121         # total length after cut

EIs_cut   = np.empty((C, L, N), dtype=EIs.dtype)
main_chan = np.empty(N, dtype=np.int32)   # optional: keep for book-keeping
src_t0    = np.empty(N, dtype=np.int32)   # optional: original peak index

for i in range(N):
    ei = EIs[:, :, i]                     # (C, T) for this unit

    # main channel = channel with the most negative value across time
    ch = np.argmin(ei.min(axis=1))
    t0 = int(np.argmin(ei[ch]))           # time index of that negative peak

    # compute desired window [start, start+L)
    start = t0 - ci
    stop  = start + L

    # pad if window falls outside [0, T)
    pre  = max(0, -start)
    post = max(0,  stop - T)
    if pre or post:
        ei = np.pad(ei, ((0, 0), (pre, post)), mode='constant')

    # slice from the (possibly) padded array; ensures peak is at index `ci`
    s = start + pre
    EIs_cut[:, :, i] = ei[:, s:s+L]

    main_chan[i] = ch
    src_t0[i]    = t0

# quick sanity check on a few units:
# assert all(EIs_cut[main_chan[k], :, k].argmin() == ci for k in range(min(N, 50)))

EIs = EIs_cut   # no copy; just rebinds the name
del EIs_cut     # drop the extra name
# optional, to eagerly return memory to Python’s allocator:
import gc; gc.collect()


# Select significant channels for proto EI

In [ ]:
unit_id  = 188-1 # 

final_ei_full = EIs[:, :, unit_id]


def _roll_zero_all(ei: np.ndarray, lag: int) -> np.ndarray:
    """Zero-padded roll (no wrap), identical to the helper in collision_utils."""
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :-lag]
    elif lag < 0:
        out[:, :lag] = ei[:, -lag:]
    else:
        out[:] = ei
    return out

def select_channels_from_ei(final_ei_full: np.ndarray, *, center_index: int = 40):
    """
    Returns (sel_ch, ref_channel, ei_rms) exactly as scan_continuous_harm_regions would choose them.
    final_ei_full: [C, T] EI (any centering).
    """
    # Re-center EI so the most negative trough on the strongest channel lands at center_index
    trough_vals_all = final_ei_full.min(axis=1)
    ref_channel = int(np.argmin(trough_vals_all))
    trough_idx  = int(np.argmin(final_ei_full[ref_channel]))
    shift = int(center_index - trough_idx)
    ei = _roll_zero_all(final_ei_full, shift)

    # Recompute ref and per-channel EI RMS after rolling
    trough_vals = ei.min(axis=1)
    ref_channel = int(np.argmin(trough_vals))
    ei_rms = np.sqrt(np.mean(ei**2, axis=1))



    return ref_channel, ei_rms

# Compute selection
ref_ch, ei_rms = select_channels_from_ei(final_ei_full, center_index=40)

robust_mean_rms = np.median(ei_rms) 
robust_std = np.median(np.abs(ei_rms - np.median(ei_rms))) * 1.4826
fac = (5.0-robust_mean_rms)/robust_std
tmp = 50
reducer_factor = 0.0
shallowest_peak = 0
while (tmp>20) or (shallowest_peak>-20):
    ch_selection_thr = robust_mean_rms + robust_std*fac + reducer_factor
    sel_mask = (ei_rms >= float(ch_selection_thr))
    sel_mask[ref_ch] = True
    sel_ch_scan = np.flatnonzero(sel_mask).astype(int)
    tmp = len(sel_ch_scan)
    neg_peak = np.min(final_ei_full[ref_ch, :])
    neg_peaks = np.min(final_ei_full[sel_ch_scan, :], axis=1)  # (N_channels,)
    shallowest_peak = np.max(neg_peaks)  # least negative of the negative peaks
    shallowest_peak_idx = np.argmax(neg_peaks)
    channel_with_shallowest_peak = sel_ch_scan[shallowest_peak_idx]
    reducer_factor = reducer_factor+1



print(f"Ref ch {ref_ch}, peak {neg_peak:.2f}. Shallowest: {shallowest_peak:.2f} on ch {channel_with_shallowest_peak}. Mean: {robust_mean_rms:.3f}, Robust STD: {robust_std:.3f}, Threshold: {ch_selection_thr:.3f}, Total {len(sel_ch_scan)} channels selected; reducer {reducer_factor}")

# 1. Get indices of top 10 channels by RMS
top_10_idx = np.argsort(ei_rms)[-7:]  # sorted ascending, so take last 10
# 2. Get indices of all channels above threshold
above_thr_idx = np.flatnonzero(ei_rms > 10.0)
# 3. Always include the reference channel
ref_idx = np.array([ref_ch])
# 4. Combine all selected indices
sel_ch_scan = np.unique(np.concatenate([top_10_idx, above_thr_idx, ref_idx])).astype(int)


print(f"Ref ch {ref_ch}, Mean: {robust_mean_rms:.3f}, Robust STD: {robust_std:.3f}, Threshold: {ch_selection_thr:.3f}, Total {len(sel_ch_scan)} channels selected")

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    final_ei_full,
    positions=ei_positions,
    ref_channel=ref_ch,
    scale=70.0,
    ax=ax,
    colors="black",
    alpha=1.0,
    linewidth=0.5,
    box_height=1.0,
    box_width=50.0,
    aspect=1.0,
)
ax.set_title('Original EI (KS)')
plt.tight_layout()
plt.show()

# Plot
plt.figure(figsize=(12, 4))
plt.plot(ei_rms, 'k.-', lw=1, ms=6, label='EI RMS (μV)')
plt.axhline(ch_selection_thr, color='gray', ls='--', label='chan_thr = 10.0')
plt.axvline(ref_ch, color='r', ls='-', lw=2, label=f'ref = {ref_ch}')
plt.scatter(sel_ch_scan, ei_rms[sel_ch_scan], color='b', s=30, zorder=3, label='selected')
plt.xlabel('Channel index')
plt.ylabel('RMS amplitude (μV)')
plt.title('Per-channel EI RMS and selection threshold')
plt.legend()
plt.tight_layout()
plt.show()
print(sel_ch_scan)

## Continuous scan

In [ ]:
results = scan_continuous_harm_regions(
    raw_data=raw_mod.T,
    final_ei_full=final_ei_full,
    start_sample=0,
    stop_sample=total_samples-200,#10_000_000,#total_samples,  # full length - will take a while. Can limit for testing 
    mean_thr=-1.0,
    chan_thr=ch_selection_thr,
    ref_thr=-3.0,
    center_index=40,
    region_max_len=20,
    amp_ratio_min=0.5,   # NEW
    amp_ratio_max=1.5,   # NEW
)

harm = results["harm_at_best"]
n_sel, n_regions = harm.shape
print(f"Spikes: {n_regions}, Channels: {n_sel}")

import os, gzip, pickle

base_dir = '/Volumes/Lab/Users/alexth/axolotl/201711290'

stem     = f'cont_scan_results_ks_data004_unit{unit_id}'
pkl_path = os.path.join(base_dir, stem + '.pkl.gz')

with gzip.open(pkl_path, 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print("saved ->", pkl_path)

# later, load:
# with gzip.open(pkl_path, 'rb') as f:
#     results = pickle.load(f)


### select spikes with at least 20samples between - logic meh! check!

In [ ]:
# results from scan_continuous_harm_regions(...)
regs = results["regions"]
best_idx = np.asarray([r["best_idx"] for r in regs], dtype=np.int64)
scores  = np.asarray([r["mean_delta"] for r in regs], dtype=np.float32)  # lower is better

min_sep = 20  # samples
order = np.argsort(scores)  # best first (most negative)
keep = np.ones(len(regs), dtype=bool)

for i in order:
    if not keep[i]:
        continue
    too_close = np.where(np.abs(best_idx - best_idx[i]) < min_sep)[0]
    too_close = too_close[too_close != i]
    keep[too_close] = False

spike_times = best_idx[keep]

print(len(spike_times))

## Plot scan results

In [ ]:

# 1) Plot compact harm map (ΔRMS per selected channel at each region’s best sample)
harm = results["harm_at_best"]             # (n_sel, n_regions)
sel_ch = results["selected_channels"]      # (n_sel,)
ref_ch = int(results["ref_channel"])      # integer channel id
v = np.nanpercentile(np.abs(harm), 95) if harm.size else 1.0


# harm: (n_channels, n_regions)  ->  X: (n_regions, n_channels)
X = harm.T

# Center features (channels). This “concatenates all channels” in the sense that
# every region’s feature vector is the ΔRMS across *all* selected channels.
Xc = X - X.mean(axis=0, keepdims=True)

# Optional (uncomment to z-score channels if you want equal weighting):
# Xc = Xc / (Xc.std(axis=0, ddof=1, keepdims=True) + 1e-12)

# PCA via SVD
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
scores = U * S  # rows = regions, cols = PCs
pc1, pc2 = scores[:, 0], scores[:, 1]
evr = (S**2) / (S**2).sum()  # explained variance ratio

# Plot PC1 vs PC2
plt.figure(figsize=(4,4))
plt.scatter(pc1, pc2, s=6, alpha=0.6)
plt.axhline(0, linestyle='--', linewidth=0.8)
plt.axvline(0, linestyle='--', linewidth=0.8)
plt.xlabel(f"PC1  ({evr[0]*100:.1f}% var)")
plt.ylabel(f"PC2  ({evr[1]*100:.1f}% var)")
plt.title("PCA of harm map (regions as points; features = channels)")
plt.tight_layout()
plt.show()


# ΔRMS histograms per channel (grid of small subplots)

# Threshold lines for reference (adjust if you used different gates)
REF_THR  = -5.0
CHAN_THR = 10.0

if harm.size == 0:
    print("harm_at_best is empty; nothing to plot.")
else:
    n_sel, n_regions = harm.shape

    # Robust global x-range so all histograms share axes
    all_vals = harm.ravel()
    x_min, x_max = np.nanpercentile(all_vals, [1, 99])
    if not np.isfinite(x_min) or not np.isfinite(x_max) or x_min == x_max:
        x_min, x_max = float(np.nanmin(all_vals)), float(np.nanmax(all_vals))
        if x_min == x_max:
            x_min, x_max = x_min - 1.0, x_max + 1.0

    bins = 100

    # Grid size ~ square
    cols = int(np.ceil(np.sqrt(n_sel)))
    rows = int(np.ceil(n_sel / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.3, rows * 1.8), sharex=True, sharey=True)
    axes = np.atleast_2d(axes)

    for i in range(rows * cols):
        ax = axes.flat[i]
        if i < n_sel:
            ch_id = int(sel_ch[i])
            x = harm[i, :]  # ΔRMS samples for this channel across regions

            ax.hist(x, bins=bins, range=(x_min, x_max), histtype='stepfilled', alpha=0.7)
            ax.axvline(0.0, color='black', linestyle='--', linewidth=0.8)
            ax.axvline(CHAN_THR, color='red', linestyle=':', linewidth=0.8)

            if ch_id == ref_ch:
                ax.axvline(REF_THR, color='magenta', linestyle='-.', linewidth=0.8)
                ax.set_title(f"{ch_id} (ref)", fontsize=9)
            else:
                ax.set_title(f"{ch_id}", fontsize=9)

            ax.tick_params(labelsize=8)
        else:
            ax.axis('off')

    fig.suptitle("ΔRMS per channel (harm at best samples)", y=1.02)
    plt.tight_layout()
    plt.show()



plt.figure(figsize=(20, 6))
im = plt.imshow(harm, aspect='auto', cmap='coolwarm', vmin=-v, vmax=v, interpolation='nearest')
plt.colorbar(im, label='ΔRMS')
plt.yticks(np.arange(len(sel_ch)), sel_ch)
plt.xlabel('Region index')
plt.ylabel('Channel (selected)')
plt.title('Harm map at best samples (compact)')
plt.tight_layout()
plt.show()

# 2) Recompute EI from the best indices
best_indices = np.array([r["best_idx"] for r in results["regions"]], dtype=np.int64)

C, T = final_ei_full.shape
ref_ch = results["ref_channel"]
ci     = results["center_index"]
pre  = -ci
post = T - ci - 1
selected_channels = np.arange(C, dtype=int)  # full EI

# spike_times=np.asarray([r["best_idx"] for r in results["regions"]], dtype=np.int64)


# results from scan_continuous_harm_regions(...)
regs = results["regions"]
best_idx = np.asarray([r["best_idx"] for r in regs], dtype=np.int64)
scores  = np.asarray([r["mean_delta"] for r in regs], dtype=np.float32)  # lower is better

min_sep = 20  # samples
order = np.argsort(scores)  # best first (most negative)
keep = np.ones(len(regs), dtype=bool)

for i in order:
    if not keep[i]:
        continue
    too_close = np.where(np.abs(best_idx - best_idx[i]) < min_sep)[0]
    too_close = too_close[too_close != i]
    keep[too_close] = False

spike_times = best_idx[keep]


print(f"Getting snippets, {len(spike_times)}")
# raw_data is (total_samples, 512) — matches extract_snippets_fast_ram
snips, valid_times = extract_snippets_fast_ram(
    raw_data=raw_mod,
    spike_times=spike_times,
    window=(pre, post),
    selected_channels=selected_channels,
)
# snips: (C, T, N). Average over spikes → EI
# ei_from_best = snips.mean(axis=2).astype(np.float32)

print("Finished snippets")

# cap EI computation to <=5k spikes, picked uniformly at random (repeatable)
N = snips.shape[2]
MAXN = 5000
if N > MAXN:
    rng = np.random.default_rng(12345)  # <- set your seed here
    sel = rng.choice(N, size=MAXN, replace=False)
else:
    sel = np.arange(N)

ei_from_best = snips[:, :, sel].mean(axis=2).astype(np.float32)


# 3) Overlay recomputed EI with original (both recentred so trough aligns at sample 40)
ei_orig_centered  = recenter_ei_to_ref_trough(final_ei_full, center_index=results["center_index"])
ei_best_centered  = recenter_ei_to_ref_trough(ei_from_best,    center_index=results["center_index"])


# Compute selection
ref_ch_best, ei_rms = select_channels_from_ei(ei_best_centered, center_index=40)

robust_mean_rms = np.median(ei_rms) 
robust_std = np.median(np.abs(ei_rms - np.median(ei_rms))) * 1.4826
ch_best_threshold = robust_mean_rms + robust_std*3.5


sel_mask = (ei_rms >= float(ch_best_threshold))
sel_mask[ref_ch_best] = True
sel_ch_best = np.flatnonzero(sel_mask).astype(int)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(ei_rms, 'k.-', lw=1, ms=6, label='EI RMS (μV)')
plt.axhline(ch_best_threshold, color='gray', ls='--', label='chan_thr = 10.0')
plt.axvline(ref_ch_best, color='r', ls='-', lw=2, label=f'ref = {ref_ch_best}')
plt.scatter(sel_ch_best, ei_rms[sel_ch_best], color='b', s=30, zorder=3, label='selected')
plt.xlabel('Channel index')
plt.ylabel('RMS amplitude (μV)')
plt.title('Per-channel EI RMS and selection threshold FOR BEST EI')
plt.legend()
plt.tight_layout()
plt.show()
print(sel_ch)


fig = plt.figure(figsize=(20, 12))
ax = fig.add_subplot(111)
pew.plot_ei_waveforms(
    [ei_orig_centered, ei_best_centered],
    positions=ei_positions,
    ref_channel=ref_ch,
    scale=70.0,
    ax=ax,
    colors=['green', 'red'],
    alpha=[0.9, 0.9],
    linewidth=[0.6, 0.6],
    box_height=1.0,
    box_width=50.0,
    aspect=1.0,
)
ax.set_title('EI overlay: original (green) vs recomputed from best indices (red)')
plt.tight_layout()
plt.show()



In [ ]:
# ISI / rate params
sample_rate_hz = 20_000
bin_width_ms   = 0.5
max_ms         = 300.0
dt_ms          = 1000.0
sigma_ms       = 2500.0

# PSTH params
WIN_SEC = 0.5
BIN_MS  = 1.0
bin_s   = BIN_MS / 1000.0
edges   = np.arange(0.0, WIN_SEC + bin_s, bin_s)
centers = edges[:-1] + 0.5 * bin_s
nbins   = len(centers)

spikes_sec = spike_times / float(sample_rate_hz)

def simple_psth(spikes_sec, onsets_sec):
    onsets_sec = np.asarray(onsets_sec, dtype=np.float64)
    onsets_sec = onsets_sec[np.isfinite(onsets_sec)]
    if onsets_sec.size == 0:
        return None, 0
    counts = np.zeros(nbins, dtype=np.float64)
    for onset in onsets_sec:
        i0 = np.searchsorted(spikes_sec, onset, side="left")
        i1 = np.searchsorted(spikes_sec, onset + WIN_SEC, side="left")
        rel = spikes_sec[i0:i1] - onset
        if rel.size == 0:
            continue
        idx = (rel / bin_s).astype(np.int64)
        idx = idx[(idx >= 0) & (idx < nbins)]
        np.add.at(counts, idx, 1)
    ntr = int(onsets_sec.size)
    rate_hz = counts / (ntr * bin_s)  # uniform divide by N trials everywhere + constant bin width
    return rate_hz, ntr

# pick the onset arrays you loaded (prefer truncated 30 min if present)
try:
    on_u_use = on_u30
except NameError:
    try:
        on_u_use = on_u
    except NameError:
        on_u_use = None

try:
    on_r_use = on_r30
    blk_use  = blk_r30
    tri_use  = tri_r30
except NameError:
    try:
        on_r_use = on_r
        blk_use  = blk_r
        tri_use  = tri_r
    except NameError:
        on_r_use = None

# Unique PSTH
if on_u_use is not None:
    psth_u, ntr_u = simple_psth(spikes_sec, on_u_use)
    plt.figure(figsize=(12, 3))
    plt.plot(centers * 1000.0, psth_u, lw=1.5)
    plt.xlabel("time from onset (ms)")
    plt.ylabel("Hz")
    plt.title(f"Unique PSTH (fixed 0.5s, N={ntr_u} trials)")
    plt.xlim(0, 500)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print(f"[CH {ch}] No on_u30/on_u found; skipping unique PSTH.")

# Repeat raster + Repeat PSTH
if on_r_use is not None:
    # repeat raster (ticks), show only first 3 blocks present + zoom first 10s
    on_r_use = np.asarray(on_r_use, dtype=np.float64)
    blk_use  = np.asarray(blk_use,  dtype=np.int64)
    tri_use  = np.asarray(tri_use,  dtype=np.int64)

    # --- show only first 3 blocks that exist in this truncated window ---
    blocks_to_show = np.sort(np.unique(blk_use))[:3]
    # --- zoom to first 10 seconds on rectified axis ---
    TMAX_SEC = 10.0
    KMAX = int(np.floor(TMAX_SEC / WIN_SEC))  # 10s / 0.5s = 20 tiles

    mask = np.isfinite(on_r_use) & np.isin(blk_use, blocks_to_show) & (tri_use < KMAX)
    on_m  = on_r_use[mask]
    blk_m = blk_use[mask]
    tri_m = tri_use[mask]

    # collect spike x-positions per block, then draw as vertical ticks
    spikes_sec = np.asarray(spikes_sec, dtype=np.float64)
    spikes_sec = np.sort(spikes_sec)

    plt.figure(figsize=(20, 4))

    for b in blocks_to_show:
        xs = []
        sel = np.where(blk_m == b)[0]
        for onset, k in zip(on_m[sel], tri_m[sel]):
            i0 = np.searchsorted(spikes_sec, onset, side="left")
            i1 = np.searchsorted(spikes_sec, onset + WIN_SEC, side="left")
            rel = spikes_sec[i0:i1] - onset
            if rel.size:
                xs.append(k * WIN_SEC + rel)
        if len(xs):
            x = np.concatenate(xs)
            plt.vlines(x, b - 0.45, b + 0.45, linewidth=0.8)  # full-height-ish ticks

    # light onset grid, only for what we show
    for k in range(KMAX):
        plt.axvline(k * WIN_SEC, color="r", lw=0.7, alpha=0.7)

    plt.xlabel("rectified time within repeats segment (s)")
    plt.ylabel("block")
    plt.title(f"Repeat raster (first 3 blocks, ticks, 0–{TMAX_SEC:.0f}s)")
    plt.ylim(blocks_to_show.min() - 0.5, blocks_to_show.max() + 0.5)
    plt.xlim(0, TMAX_SEC)
    plt.tight_layout()
    plt.show()

    psth_r, ntr_r = simple_psth(spikes_sec, on_r_use)
    plt.figure(figsize=(12, 3))
    plt.plot(centers * 1000.0, psth_r, lw=1.5)
    plt.xlabel("time from onset (ms)")
    plt.ylabel("Hz")
    plt.title(f"Repeat PSTH (fixed 0.5s, N={ntr_r} trials)")
    plt.xlim(0, 500)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print(f"[CH {ch}] No on_r30/on_r found; skipping repeat raster/PSTH.")

# ISI 
try:
    t_ms = (spike_times / float(sample_rate_hz)) * 1000.0
    isi_ms = np.diff(t_ms)

    bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
    counts, edges = np.histogram(isi_ms, bins=bins)
    centers = edges[:-1] + 0.5 * bin_width_ms

    dt = dt_ms / 1000.0
    sigma_samples = sigma_ms / dt_ms
    spikes_sec = spike_times / float(sample_rate_hz)
    total_duration = float(spikes_sec.max() + 0.1) if spikes_sec.size > 0 else 1.0
    time_vector = np.arange(0.0, total_duration, dt)
    rate_counts, _ = np.histogram(spikes_sec, bins=np.append(time_vector, total_duration))
    rate = gaussian_filter1d(rate_counts / dt, sigma=sigma_samples)

    fig, axes = plt.subplots(1, 2, figsize=(20, 3))
    axes[0].plot(centers, counts, lw=1.5)
    axes[0].set_xlim(0, max_ms)
    axes[0].set_xlabel("ISI (ms)")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"ISI hist (bin={bin_width_ms} ms, max={max_ms} ms)\nN pairs={isi_ms.size}")

    axes[1].plot(time_vector, rate)
    axes[1].set_xlim(0, total_duration)
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Hz")
    axes[1].set_title("Smoothed firing rate")

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"[CH {ch}] ISI/rate plot failed: {e}")

# RECURSION

In [ ]:

dprime_thr = 2.
rms_thresh = 5.0
min_spikes = 500

# ---- params ----
sample_rate_hz = 20_000

# PSTH params
WIN_SEC = 0.5
BIN_MS  = 1.0
bin_s   = BIN_MS / 1000.0
edges   = np.arange(0.0, WIN_SEC + bin_s, bin_s)
centers = edges[:-1] + 0.5 * bin_s
nbins   = len(centers)

# ISI params
bin_width_ms = 0.5
max_ms       = 300.0

def simple_psth(spikes_sec, onsets_sec):
    on = np.asarray(onsets_sec, dtype=np.float64)
    on = on[np.isfinite(on)]
    if on.size == 0: 
        return None, 0
    counts = np.zeros(nbins, dtype=np.float64)
    for onset in on:
        i0 = np.searchsorted(spikes_sec, onset, side="left")
        i1 = np.searchsorted(spikes_sec, onset + WIN_SEC, side="left")
        rel = spikes_sec[i0:i1] - onset
        if rel.size:
            idx = (rel / bin_s).astype(np.int64)
            idx = idx[(idx >= 0) & (idx < nbins)]
            np.add.at(counts, idx, 1)
    ntr = int(on.size)
    return counts / (ntr * bin_s), ntr

def kmeans_split_diagnostics_subset(snips_sub, E_sub, *, channel_ids, subsample_n=8000, seed=0, **kwargs):
    """
    Wrapper: run your existing kmeans_split_diagnostics on a subset of spikes.
    snips_sub: [C_sel, L, N] for diagnostics
    E_sub    : [C_sel, L]
    channel_ids: global channel ids (as you already pass)
    """
    import numpy as np
    N = snips_sub.shape[2]
    if (subsample_n is None) or (N <= int(subsample_n)):
        use_idx = np.arange(N, dtype=int)
    else:
        rng = np.random.default_rng(seed)
        use_idx = rng.choice(N, size=int(subsample_n), replace=False)
    return kmeans_split_diagnostics(
        snips_sub[:, :, use_idx], E_sub, channel_ids=channel_ids, **kwargs
    )



def _p2p_window(E, ci=40, win=20):
    C, L = E.shape
    lo = max(0, int(ci) - int(win))
    hi = min(L, int(ci) + int(win) + 1)
    seg = E[:, lo:hi]
    return np.nanmax(seg, axis=1) - np.nanmin(seg, axis=1), lo, hi

def select_discriminative_channels(E0, E1, *, ei_positions=None,
                                   ci=40, win=20,
                                   p2p_gate=20.0,     # require signal on at least one EI
                                   cos_thr=0.985,     # exclude channels where |cos| >= cos_thr (too similar)
                                   max_keep=30,       # hard cap for speed
                                   ensure_mains=True):
    """
    Pick channels that matter for separating E0 vs E1:
      - must have signal: max(p2p0,p2p1) >= p2p_gate
      - must differ: |cos(E0c, E1c)| < cos_thr (computed in window)
    Rank by (p2p0+p2p1)*(1-|cos|), keep up to max_keep. Always include main channels.
    Returns: sorted unique channel indices (np.int32)
    """
    E0 = np.asarray(E0, dtype=np.float32); E1 = np.asarray(E1, dtype=np.float32)
    assert E0.shape == E1.shape, "E0/E1 must have same [C,L]"
    C, L = E0.shape

    p2p0, lo, hi = _p2p_window(E0, ci=ci, win=win)
    p2p1, _,  _  = _p2p_window(E1, ci=ci, win=win)

    X = E0[:, lo:hi]; Y = E1[:, lo:hi]
    # per-channel cosine in window (NaN-safe)
    num = np.sum(X*Y, axis=1)
    den = np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1) + 1e-12
    cos = num / den
    cos = np.clip(cos, -1.0, 1.0)
    dis = 1.0 - np.abs(cos)  # larger = more different

    sig = np.maximum(p2p0, p2p1)
    mask = (sig >= float(p2p_gate)) & (np.abs(cos) < float(cos_thr))

    score = (p2p0 + p2p1) * dis
    idx = np.flatnonzero(mask)
    if idx.size:
        order = idx[np.argsort(-score[idx])]
        keep = order[:int(max_keep)]
    else:
        keep = np.array([], dtype=int)

    if ensure_mains:
        m0 = int(np.nanargmin(np.nanmin(E0, axis=1)))
        m1 = int(np.nanargmin(np.nanmin(E1, axis=1)))
        keep = np.unique(np.concatenate([keep, np.array([m0, m1], dtype=int)]))

    return keep.astype(np.int32)

# Use exactly the channels you pass; no internal re-selection.
def harm_map_locked(E, SN, idx, *, lag_radius=0, title="", sort_by_ptp=True):
    import numpy as np
    from collision_utils import compute_harm_map_noamp, plot_harm_heatmap

    E = np.asarray(E, dtype=np.float32)            # [M, L]   (M is your preselected channels)
    SN = np.asarray(SN, dtype=np.float32)          # [M, L, Nall]
    idx = np.asarray(idx, dtype=int)

    M = E.shape[0]
    res = compute_harm_map_noamp(
        E, SN[:, :, idx],
        # LOCK the set: min == max == M and no thresholding
        p2p_thr=-1e9,
        max_channels=M,
        min_channels=M,
        lag_radius=int(lag_radius),
        weight_by_p2p=True,
        weight_beta=0.7,
        # if the impl supports it, keeping main-channel flags off makes no difference
        force_include_main=False
    )

    # Sanity: verify it didn’t shrink the channel dimension
    try:
        H = res["harm_matrix"]
        if H.shape[0] != M:
            print(f"[warn] harm_map_locked: expected M={M} channels, got {H.shape[0]}. "
                  "That means the function still reselected internally.")
    except Exception:
        pass

    plot_harm_heatmap(res, field="harm_matrix", sort_by_ptp=bool(sort_by_ptp), title=title)
    return res


def recursive_split_and_visualize(
    snips,                    # [C, L, N]
    ei_centered,              # [C, L]
    ei_positions,
    *,
    rms_thr=6.0,
    dprime_thr=3.0,
    min_per_cluster=10,
    n_init=8,
    max_iter=60,
    lag_radius=0,
    max_diag_channels=30,
    diag=False,               # NEW: skip heavy diagnostics by default
    min_spikes_leaf=20,
    # --- NEW: similarity stop (replaces any main-channel gating) ---
    sim_stop_cos_thr=0.992,           # stop if children cosine >= this
    sim_stop_rms_gate=5.0,            # channels used if RMS>gate in either child
    sim_stop_best_align_lag=6,        # best global shift in ±lag samples
    # (legacy args below are ignored if still present in call sites)
    recurse_only_if_main_eq_ref=False,
    recurse_gate_amp_delta=None,
    branch_prefer_ref=False,
    ref_ch=None,
    ci=40,
    depth=0,
    max_depth=6,
    pool_idx=None,
    # --- NEW: AB shard diff-gate knobs ---
    abdiff_rms_gate=5.0,      # union RMS gate to include channel in comparison
    abdiff_mag_gate=100.0,    # abs(signed Euclidean) must exceed this to count
    abdiff_plot=True          # keep the bar plot
):


    """
    Recursively run your k=2 channel split + harm-map diagnostics until no further split is possible.
    Keeps *all* the same plots at each node. Returns a list of leaf groups with absolute spike indices.
    """
    import numpy as _np
    import matplotlib.pyplot as _plt
    import plot_ei_waveforms as pew


    # ---------- book-keeping ----------
    indent = "  " * depth
    _print = lambda s: print(f"{indent}{s}")

    print("Starting new recursion")


    if len(plt.get_fignums()) > 20:
        plt.close('all')

    # --- STA helper: absolute SAMPLES from root `spike_times` via pool_idx ---
    if 'spike_times' in globals():
        _spike_times_root = _np.asarray(globals()['spike_times'], dtype=_np.int64).ravel()
    else:
        _spike_times_root = None  # we'll warn if missing

    def _sta_samples(local_idx):
        if _spike_times_root is None:
            raise NameError("`spike_times` (absolute samples) not found in globals.")
        # local child indices -> root indices -> absolute samples
        return _spike_times_root[pool_idx[_np.asarray(local_idx, dtype=int)]]

    def _sta_samples_abs(abs_idx):
        if _spike_times_root is None:
            raise NameError("`spike_times` (absolute samples) not found in globals.")
        return _spike_times_root[_np.asarray(abs_idx, dtype=int)]

    if pool_idx is None:
        pool_idx = _np.arange(snips.shape[2], dtype=int)

    pool_idx = _np.asarray(pool_idx, dtype=int).ravel()
    N_here = int(pool_idx.size)

    if N_here == 0:
        _print(f"[stop] depth={depth} | N=0 (empty)")
        return []

    # stop a branch early if too few spikes in the current pool
    if N_here < int(min_spikes_leaf):
        _print(f"[stop] depth={depth} | N={N_here} (<{min_spikes_leaf})")
        return [{"idx_abs": pool_idx.copy(),
                "ei": ei_centered.copy(),
                "depth": depth,
                "note": "min_spikes_leaf"}]

    if ref_ch is None:
        # “most negative trough” channel
        ref_ch = int(_np.argmin(ei_centered.min(axis=1)))



    # ---------- (A) diagnostics (optional) ----------
    if diag:
        rms = _np.sqrt(_np.mean(ei_centered**2, axis=1)).astype(_np.float32)
        channels = _np.flatnonzero(rms > rms_thr).astype(int)
        order = channels[_np.argsort(-rms[channels])]               # global ids sorted by RMS
        top = order[:int(max_diag_channels)]                        # cap to top-K

        _print(f"[mix-check] {channels.size} channels pass RMS>{rms_thr:g}; "
            f"using top {top.size} by RMS: {top[:20]}{' ...' if top.size>20 else ''}")

        if top.size:
            _ = kmeans_split_diagnostics_subset(
                snips[channels, :, :],
                ei_centered[channels, :],
                channel_ids=channels,
                subsample_n=8000, seed=0,
                rms_thr=0.0, n_init=8, max_iter=60, ncols=4,
                title_prefix=f"post-explain (depth {depth})"
            )
        else:
            _print("[mix-check] No channels pass RMS; skipping k-means diagnostics.")
    else:
        _print("[mix-check] diag=False → skip k-means diagnostics.")



    # ---------- (B) try the split on the current pool ----------
    print("trying to split")
    res_split = split_first_good_channel_subset_and_harm(
        snips, ei_centered, ei_positions,
        rms_thr=rms_thr, dprime_thr=dprime_thr, min_per_cluster=min_per_cluster,
        n_init=n_init, max_iter=max_iter, lag_radius=lag_radius,
        max_diag_channels=max_diag_channels, subsample_n=8000, subsample_seed=0,
        ei_subsample_n=1000, ei_subsample_seed=0,    active_idx=pool_idx,       # <-- NEW
        assign_block=8192          # <-- NEW (tune)
    )
    print("finished splitting")   

    # Base case: no valid split or depth cap hit
    if res_split is None or depth >= max_depth:
        _print(f"[stop] depth={depth} | N={N_here} (no further split)")
        return [{"idx_abs": pool_idx.copy(), "ei": ei_centered.copy(), "depth": depth}]

    # ---------- (B2) AB-shard decision via signed per-channel differences ----------
    # Compare EI0 vs EI1 on the UNION of channels with RMS > abdiff_rms_gate.
    # Signed Euclidean per channel; sign = + if EI0 has deeper negative trough, − if EI1.

    E0 = _np.asarray(res_split['EI0'], dtype=_np.float32)
    E1 = _np.asarray(res_split['EI1'], dtype=_np.float32)

    fig, ax = plt.subplots(figsize=(15, 12))
    pew.plot_ei_waveforms(
        [E0, E1], ei_positions,
        ref_channel=ref_ch,
        colors=["black", "red"],
        scale=70.0, box_height=1.0, box_width=50.0,
        ax=ax
    )
    plt.title(f"EI overlay for AB shard comparison")
    plt.tight_layout()
    plt.show()

    # --- size guard: skip AB-check if both children are large ---
    _N0 = int(_np.asarray(res_split['idx0']).size)
    _N1 = int(_np.asarray(res_split['idx1']).size)
    abdiff_n_guard = 400
    _do_ab = not ((_N0 >= abdiff_n_guard) and (_N1 >= abdiff_n_guard))

    _do_ab = False

    if not _do_ab:
        _print(f"[AB-check] size guard: N0={_N0}, N1={_N1} ≥ {abdiff_n_guard} → skip AB-check (treat as two cells).")
    else:
        # Shape guard and union RMS gate
        C = min(E0.shape[0], E1.shape[0]); L = min(E0.shape[1], E1.shape[1])
        E0 = E0[:C, :L]; E1 = E1[:C, :L]
        rms0 = _np.sqrt(_np.mean(E0**2, axis=1)).astype(_np.float32)
        rms1 = _np.sqrt(_np.mean(E1**2, axis=1)).astype(_np.float32)
        use_mask = (rms0 > float(abdiff_rms_gate)) | (rms1 > float(abdiff_rms_gate))

        if use_mask.any():
            # Vectorized signed distances:
            # sign = +1 if EI0 has deeper (more negative) trough than EI1 on that channel
            n0 = _np.abs(_np.min(E0, axis=1))
            n1 = _np.abs(_np.min(E1, axis=1))
            sign = _np.sign(n0 - n1).astype(_np.float32)  # +1, 0, or -1
            diff_mag = _np.linalg.norm(E0 - E1, axis=1).astype(_np.float32)
            diffs = diff_mag * sign  # signed per-channel distance
            keep_idx = _np.flatnonzero(use_mask & (_np.abs(diffs) >= float(abdiff_mag_gate)))
        else:
            keep_idx = _np.array([], dtype=int)
            diffs = _np.zeros(C, dtype=_np.float32)

        if diag and abdiff_plot and keep_idx.size:
            y = diffs[keep_idx]
            colors = _np.where(y >= 0, 'tab:red', 'tab:green')
            _plt.figure(figsize=(14, 4))
            _plt.bar(keep_idx, y, color=colors, width=0.8)
            _plt.axhline(0, color='k', linewidth=1)
            _plt.xlabel("Channel"); _plt.ylabel("Signed difference (||EI0−EI1||)")
            _plt.title(f"[depth {depth}] AB-check (RMS>{abdiff_rms_gate}, |diff|>{abdiff_mag_gate})")
            _plt.tight_layout(); _plt.show()

        # Decision:
        if keep_idx.size:
            vals = diffs[keep_idx]
            pos_any = bool(_np.any(vals > 0))
            neg_any = bool(_np.any(vals < 0))
            if pos_any and neg_any:
                _print("[AB-check] mixed signs after gate → looks like TWO CELLS; proceeding with split.")
            else:
                sign_label = "+only (EI0 deeper)" if pos_any else "−only (EI1 deeper)"
                _print(f"[AB-check] single-sign ({sign_label}) after gate → AB SHARD; discard split and stop.")
                return [{
                    "idx_abs": pool_idx.copy(),
                    "ei": ei_centered.copy(),
                    "depth": depth,
                    "note": f"ab_shard_stop[{sign_label}]"
                }]
        else:
            _print("[AB-check] no channels survived diff magnitude gate; cannot decide here (continue).")






    # Fast harm-map & plots on the discriminative subset only
    # If you already built a discriminative set `sel_ch`, use that.
    # Otherwise, lock to every channel you intend to evaluate (e.g., union set or top-K).
        
    # Fast harm-map & plots (optional)
    if diag:

            # ---------- (C) harm maps + classification (as in your code) ----------
        sel_ch = select_discriminative_channels(
            res_split['EI0'], res_split['EI1'],
            ei_positions=ei_positions,
            ci=int(ci), win=20,
            p2p_gate=20.0,     # tweak if your EIs are smaller/bigger
            cos_thr=0.985,     # higher -> stricter "too similar", fewer channels
            max_keep=30,       # main speed lever
            ensure_mains=True
        )
        if sel_ch.size == 0:
            _print("[harm-fast] no discriminative channels found; falling back to mains only")
            sel_ch = select_discriminative_channels(
                res_split['EI0'], res_split['EI1'], ci=int(ci), win=20,
                p2p_gate=-1e9, cos_thr=2.0, max_keep=2, ensure_mains=True
            )
        E0s = res_split['EI0'][sel_ch, :]
        E1s = res_split['EI1'][sel_ch, :]
        SNs = snips[sel_ch, :, :]
        _ = harm_map_locked(
            E0s, SNs, res_split['idx0'],
            lag_radius=lag_radius,
            title=f"depth {depth} — Template EI0 on locked channels (M={len(sel_ch)})",
            sort_by_ptp=True
        )
        _ = harm_map_locked(
            E1s, SNs, res_split['idx1'],
            lag_radius=lag_radius,
            title=f"depth {depth} — Template EI1 on locked channels (M={len(sel_ch)})",
            sort_by_ptp=True
        )
    else:
        _print(f"[harm-fast] diag=False → skip harm maps on children ")



    # metrics = classify_two_cells_vs_ab_shard(
    #     E0s, E1s, sn_s, res_split['idx0'], res_split['idx1'],
    #     p2p_thr=-1e9, max_channels=sel_ch.size, min_channels=sel_ch.size,
    #     lag_radius=lag_radius, weight_by_p2p=True, weight_beta=0.7,
    #     rms_thr_support=10.0,
    #     asym_strong_z=2.0, asym_pure_z=1.0
    # )


    # ---------- (D) keep your EI plots ----------
    N0 = int(_np.asarray(res_split['idx0']).size)
    N1 = int(_np.asarray(res_split['idx1']).size)

    fig, ax = _plt.subplots(figsize=(15, 12))
    pew.plot_ei_waveforms(res_split['EI0'], ei_positions,
                        ref_channel=int(ref_ch), colors='black',
                        scale=70.0, box_height=1.0, box_width=50.0, ax=ax)
    ax.set_title(f"[depth {depth}] EI0 (N={N0})")
    _plt.show()

    # --- STAs for both children (ABSOLUTE SAMPLES) ---
    # Inputs
    sample_rate_hz = 20_000   # 20 kHz
    bin_width_ms   = 0.5
    max_ms         = 300.0

    try:
        t0_samp = _sta_samples_abs(res_split['idx0'])  # absolute samples
        spikes_sec = np.asarray(t0_samp, dtype=np.float64) / float(sample_rate_hz)
        
        # prefer truncated 30 min if present
        on_u_use = on_u30 if "on_u30" in globals() else (on_u if "on_u" in globals() else None)

        fig, ax = plt.subplots(1, 2, figsize=(16, 3))

        # ---- Unique PSTH ----
        if on_u_use is not None:
            psth_u, ntr_u = simple_psth(spikes_sec, on_u_use)
            if psth_u is not None:
                ax[0].plot(centers * 1000.0, psth_u, lw=1.5)
                ax[0].set_title(f"Unique PSTH (0–{WIN_SEC*1000:.0f} ms, N={ntr_u})")
                ax[0].set_xlim(0, WIN_SEC * 1000.0)
                ax[0].grid(alpha=0.25)
            else:
                ax[0].text(0.5, 0.5, "No valid unique onsets", ha="center", va="center", transform=ax[0].transAxes)
        else:
            ax[0].text(0.5, 0.5, "No on_u30/on_u found", ha="center", va="center", transform=ax[0].transAxes)
        ax[0].set_xlabel("time from onset (ms)")
        ax[0].set_ylabel("Hz")

        # ---- ISI histogram ----
        t_ms = spikes_sec * 1000.0
        isi_ms = np.diff(t_ms)
        bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
        counts, e = np.histogram(isi_ms, bins=bins)
        c = e[:-1] + 0.5 * bin_width_ms
        ax[1].plot(c, counts, lw=1.5)
        ax[1].set_title(f"ISI hist (≤{max_ms} ms, bin={bin_width_ms} ms)\nN pairs={isi_ms.size}")
        ax[1].set_xlim(0, max_ms)
        ax[1].set_xlabel("ISI (ms)")
        ax[1].set_ylabel("Count")

        plt.tight_layout()
        plt.show()
    except Exception as e:
        _print(f"[PSTH/ISI] skipped at depth {depth}: {e}")

    fig, ax = _plt.subplots(figsize=(15, 12))
    pew.plot_ei_waveforms(res_split['EI1'], ei_positions,
                        ref_channel=int(ref_ch), colors='black',
                        scale=70.0, box_height=1.0, box_width=50.0, ax=ax)
    ax.set_title(f"[depth {depth}] EI1 (N={N1})")
    _plt.show()

    # --- STAs for both children (ABSOLUTE SAMPLES) ---
    try:
        t1_samp = _sta_samples_abs(res_split['idx1'])  # absolute samples
        spikes_sec = np.asarray(t1_samp, dtype=np.float64) / float(sample_rate_hz)
        
        # prefer truncated 30 min if present
        on_u_use = on_u30 if "on_u30" in globals() else (on_u if "on_u" in globals() else None)

        fig, ax = plt.subplots(1, 2, figsize=(16, 3))

        # ---- Unique PSTH ----
        if on_u_use is not None:
            psth_u, ntr_u = simple_psth(spikes_sec, on_u_use)
            if psth_u is not None:
                ax[0].plot(centers * 1000.0, psth_u, lw=1.5)
                ax[0].set_title(f"Unique PSTH (0–{WIN_SEC*1000:.0f} ms, N={ntr_u})")
                ax[0].set_xlim(0, WIN_SEC * 1000.0)
                ax[0].grid(alpha=0.25)
            else:
                ax[0].text(0.5, 0.5, "No valid unique onsets", ha="center", va="center", transform=ax[0].transAxes)
        else:
            ax[0].text(0.5, 0.5, "No on_u30/on_u found", ha="center", va="center", transform=ax[0].transAxes)
        ax[0].set_xlabel("time from onset (ms)")
        ax[0].set_ylabel("Hz")

        # ---- ISI histogram ----
        t_ms = spikes_sec * 1000.0
        isi_ms = np.diff(t_ms)
        bins = np.arange(0.0, max_ms + bin_width_ms, bin_width_ms)
        counts, e = np.histogram(isi_ms, bins=bins)
        c = e[:-1] + 0.5 * bin_width_ms
        ax[1].plot(c, counts, lw=1.5)
        ax[1].set_title(f"ISI hist (≤{max_ms} ms, bin={bin_width_ms} ms)\nN pairs={isi_ms.size}")
        ax[1].set_xlim(0, max_ms)
        ax[1].set_xlabel("ISI (ms)")
        ax[1].set_ylabel("Count")

        plt.tight_layout()
        plt.show()
    except Exception as e:
        _print(f"[PSTH/ISI] skipped at depth {depth}: {e}")


    # ---------- (E) keep your ref-channel waveform overlays ----------
    # wfs0 = snips[ref_ch, :, :][:, res_split['idx0']]
    # wfs1 = snips[ref_ch, :, :][:, res_split['idx1']]
    # mean_wf0 = wfs0.mean(axis=1)
    # mean_wf1 = wfs1.mean(axis=1)

    # fig, axes = _plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    # ax = axes[0]
    # ax.plot(wfs0[:500], color='black', linewidth=0.25, alpha=0.08)
    # ax.plot(mean_wf0, color='red', linewidth=2.5, alpha=0.95)
    # ax.plot(mean_wf1, color='green', linewidth=2.5, alpha=0.95)
    # ax.axvline(ci, color='k', linestyle='--', linewidth=1, alpha=0.6)
    # ax.set_title(f"[depth {depth}] Ref ch {ref_ch}: IDX0 (N={wfs0.shape[1]})", fontsize=14)
    # ax.set_xlabel("Sample"); ax.set_ylabel("μV"); ax.set_ylim(-100, 100)

    # ax = axes[1]
    # ax.plot(wfs1[:500], color='black', linewidth=0.25, alpha=0.08)
    # ax.plot(mean_wf0, color='red', linewidth=2.5, alpha=0.95)
    # ax.plot(mean_wf1, color='green', linewidth=2.5, alpha=0.95)
    # ax.axvline(ci, color='k', linestyle='--', linewidth=1, alpha=0.6)
    # ax.set_title(f"[depth {depth}] Ref ch {ref_ch}: IDX1 (N={wfs1.shape[1]})", fontsize=14)
    # ax.set_xlabel("Sample"); ax.set_ylim(-100, 100)
    # _plt.tight_layout(); _plt.show()

    # ---------- (F) recurse only when it looks like two real cells ----------
    # idx0_abs = pool_idx[_np.asarray(res_split['idx0'], dtype=int)]
    # idx1_abs = pool_idx[_np.asarray(res_split['idx1'], dtype=int)]

    idx0_abs = _np.asarray(res_split['idx0'], dtype=int)
    idx1_abs = _np.asarray(res_split['idx1'], dtype=int)

    # label = metrics.get("label", "unknown")
    # _print(f"[decision] {label} | N0={idx0_abs.size}  N1={idx1_abs.size} — considering recursion")

    # --- NEW: similarity stop — if children are too similar, stop at this node ---
    c0_mid, c1_mid, cU_mid = cosine_two_eis_asym(res_split['EI0'][:,35:80], res_split['EI1'][:,35:80], rms_gate=5.0, best_align_lag=6, shared_lag=True)
    print(f"Asym cosine for mid range: {c0_mid:.3f},  {c1_mid:.3f},  {cU_mid:.3f}")
    c0, c1, cU = cosine_two_eis_asym(res_split['EI0'], res_split['EI1'], rms_gate=5.0, best_align_lag=6, shared_lag=True)
    print(f"Asym cosine for full range:: {c0:.3f},  {c1:.3f},  {cU:.3f}")
    max_cosine = np.max([c0, c1, cU, c0_mid, c1_mid, cU_mid])

    sim_cos, sim_nch, sim_lag, sim_olen = cosine_two_eis(
        res_split['EI0'], res_split['EI1'],
        rms_gate=sim_stop_rms_gate,
        use_abs=True,
        best_align_lag=sim_stop_best_align_lag
    )

    check_ei = res_split['EI0']-res_split['EI1']

    # peak-to-peak (µV) per channel
    p2p = check_ei.max(axis=1) - check_ei.min(axis=1)

    # index of largest
    ch_max = np.argmax(p2p)
    max_p2p = p2p[ch_max]
    p2p_dist = abs(np.argmax(check_ei[ch_max])-np.argmin(check_ei[ch_max]))


    _print(f"[similarity] child EIs cos={sim_cos if _np.isfinite(sim_cos) else _np.nan:.4f}; "
        f"(nch={sim_nch}, lag={sim_lag:+d}, overlapL={sim_olen}) | thr={sim_stop_cos_thr}; "
        f"p2p: {max_p2p:.2f} ADC on channel {ch_max} (ref is {ref_ch}), distance {p2p_dist}")

    if _np.isfinite(sim_cos) and (sim_cos >= float(sim_stop_cos_thr)):
        _print(f"[stop] children too similar (cos≥thr) → stop split at depth={depth}")
        return [{
            "idx_abs": pool_idx.copy(),
            "ei": ei_centered.copy(),
            "depth": depth,
            "note": f"children_cos≥thr({sim_cos:.3f}≥{sim_stop_cos_thr})"
        }]

    # if max_cosine>= float(sim_stop_cos_thr):
    #     _print(f"[stop] children too similar (max_cosine≥thr) → stop split at depth={depth}")
    #     return [{
    #         "idx_abs": pool_idx.copy(),
    #         "ei": ei_centered.copy(),
    #         "depth": depth,
    #         "note": f"children_max_cos≥thr({max_cosine:.3f}≥{sim_stop_cos_thr})"
    #     }]

    # if p2p_dist>25 or max_p2p<40:
    #     _print(f"[stop] children too similar (p2p) → stop split at depth={depth}")
    #     return [{
    #         "idx_abs": pool_idx.copy(),
    #         "ei": ei_centered.copy(),
    #         "depth": depth,
    #         "note": f"children_p2p≥thr({max_p2p:.3f},{p2p_dist})"
    #     }]


    # --- Prefer branch closer to reference template (if enabled) ---
    if branch_prefer_ref:
        cos0, nch0, lag0, _ = cosine_two_eis(
            res_split['EI0'], ei_centered,
            rms_gate=sim_stop_rms_gate,
            use_abs=True,
            best_align_lag=sim_stop_best_align_lag
        )
        cos1, nch1, lag1, _ = cosine_two_eis(
            res_split['EI1'], ei_centered,
            rms_gate=sim_stop_rms_gate,
            use_abs=True,
            best_align_lag=sim_stop_best_align_lag
        )
        _print(f"[branch] vs ref: cos0={cos0 if _np.isfinite(cos0) else _np.nan:.4f} "
            f"(nch={nch0}, lag={lag0:+d}) | "
            f"cos1={cos1 if _np.isfinite(cos1) else _np.nan:.4f} "
            f"(nch={nch1}, lag={lag1:+d})")

        # Decide which child to recurse; the other becomes a leaf
        # (tie goes to EI0)
        take_left = False
        if _np.isfinite(cos0) and _np.isfinite(cos1):
            take_left = (cos0 >= cos1)
        elif _np.isfinite(cos0) and not _np.isfinite(cos1):
            take_left = True
        elif not _np.isfinite(cos0) and _np.isfinite(cos1):
            take_left = False
        else:
            # both NaN — fall back to recursing both
            take_left = None

        if take_left is not None:
            if take_left:
                _print("[branch] recursing LEFT (closer to ref); RIGHT → leaf")
                leaves_left = recursive_split_and_visualize(
                    snips[:, :, res_split['idx0']], res_split['EI0'], ei_positions,
                    rms_thr=rms_thr, dprime_thr=dprime_thr, min_per_cluster=min_per_cluster,
                    n_init=n_init, max_iter=max_iter, lag_radius=lag_radius,
                    max_diag_channels=max_diag_channels, min_spikes_leaf=min_spikes_leaf,
                    sim_stop_cos_thr=sim_stop_cos_thr, sim_stop_rms_gate=sim_stop_rms_gate,
                    sim_stop_best_align_lag=sim_stop_best_align_lag,
                    branch_prefer_ref=branch_prefer_ref,
                    ref_ch=ref_ch, ci=ci, depth=depth+1, max_depth=max_depth,
                    pool_idx=idx0_abs
                )
                leaves_right = [{
                    "idx_abs": idx1_abs.copy(),
                    "ei": res_split['EI1'].copy(),
                    "depth": depth+1,
                    "note": "skipped_by_ref_preference"
                }]
            else:
                _print("[branch] recursing RIGHT (closer to ref); LEFT → leaf")
                leaves_left = [{
                    "idx_abs": idx0_abs.copy(),
                    "ei": res_split['EI0'].copy(),
                    "depth": depth+1,
                    "note": "skipped_by_ref_preference"
                }]
                leaves_right = recursive_split_and_visualize(
                    snips[:, :, res_split['idx1']], res_split['EI1'], ei_positions,
                    rms_thr=rms_thr, dprime_thr=dprime_thr, min_per_cluster=min_per_cluster,
                    n_init=n_init, max_iter=max_iter, lag_radius=lag_radius,
                    max_diag_channels=max_diag_channels, min_spikes_leaf=min_spikes_leaf,
                    sim_stop_cos_thr=sim_stop_cos_thr, sim_stop_rms_gate=sim_stop_rms_gate,
                    sim_stop_best_align_lag=sim_stop_best_align_lag,
                    branch_prefer_ref=branch_prefer_ref,
                    ref_ch=ref_ch, ci=ci, depth=depth+1, max_depth=max_depth,
                    pool_idx=idx1_abs
                )
            return leaves_left + leaves_right


    # --- otherwise (branch_prefer_ref False or no valid ref): recurse both as before ---

    leaves_left = recursive_split_and_visualize(
        snips, res_split['EI0'], ei_positions,
        rms_thr=rms_thr, dprime_thr=dprime_thr, min_per_cluster=min_per_cluster,
        n_init=n_init, max_iter=max_iter, lag_radius=lag_radius,
        max_diag_channels=max_diag_channels, min_spikes_leaf=min_spikes_leaf,
        sim_stop_cos_thr=sim_stop_cos_thr, sim_stop_rms_gate=sim_stop_rms_gate,
        sim_stop_best_align_lag=sim_stop_best_align_lag,branch_prefer_ref=branch_prefer_ref,
        ref_ch=ref_ch, ci=ci, depth=depth+1, max_depth=max_depth,
        pool_idx=idx0_abs
    )
    leaves_right = recursive_split_and_visualize(
        snips, res_split['EI1'], ei_positions,
        rms_thr=rms_thr, dprime_thr=dprime_thr, min_per_cluster=min_per_cluster,
        n_init=n_init, max_iter=max_iter, lag_radius=lag_radius,
        max_diag_channels=max_diag_channels, min_spikes_leaf=min_spikes_leaf,
        sim_stop_cos_thr=sim_stop_cos_thr, sim_stop_rms_gate=sim_stop_rms_gate,
        sim_stop_best_align_lag=sim_stop_best_align_lag,branch_prefer_ref=branch_prefer_ref,
        ref_ch=ref_ch, ci=ci, depth=depth+1, max_depth=max_depth,
        pool_idx=idx1_abs
    )
    return leaves_left + leaves_right



# === Split & visualize based on first good channel — *RECURSIVE* ===
leaves = recursive_split_and_visualize(
    snips, ei_best_centered, ei_positions,
    rms_thr=rms_thresh, dprime_thr=dprime_thr,
    min_per_cluster=min_spikes, n_init=3, max_iter=30, lag_radius=0,
    max_diag_channels=30, min_spikes_leaf=min_spikes,
    sim_stop_cos_thr=0.95, sim_stop_rms_gate=5.0, sim_stop_best_align_lag=6,
    branch_prefer_ref=False,
    ref_ch=int(ref_ch) if 'ref_ch' in locals() else None,
    ci=int(ci) if 'ci' in locals() else 40,
    max_depth=10, diag=False 
)
print(f"[done] recursion produced {len(leaves)} leaf groups")



In [ ]:
# === Leaf EI vs "original KS EI" overlays (cosine in title) ===

import numpy as np
import matplotlib.pyplot as plt

# Try to ensure the EI plotter exists
try:
    import plot_ei_waveforms as pew
except Exception:
    pew = None

# Try to reuse your recenter helper; fall back to a local copy if missing
try:
    from joint_utils import recenter_ei_to_ref_trough
except Exception:
    def recenter_ei_to_ref_trough(ei: np.ndarray, center_index: int = 40) -> np.ndarray:
        ei = np.asarray(ei, dtype=np.float32)
        C, T = ei.shape
        ref_ch = int(np.argmin(ei.min(axis=1)))
        trough_idx = int(np.argmin(ei[ref_ch]))
        shift = int(center_index - trough_idx)
        if shift == 0:
            return ei.copy()
        out = np.zeros_like(ei)
        if shift > 0:
            out[:, shift:] = ei[:, :T-shift]
        else:
            s = -shift
            out[:, :T-s] = ei[:, s:]
        return out

def _zero_pad_shift(ei: np.ndarray, lag: int) -> np.ndarray:
    """Zero-padded shift along time axis. lag>0 shifts right, lag<0 shifts left."""
    ei = np.asarray(ei, dtype=np.float32)
    C, T = ei.shape
    lag = int(lag)
    if lag == 0:
        return ei
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :T-lag]
    else:
        s = -lag
        out[:, :T-s] = ei[:, s:]
    return out

def _best_cosine_eis(A, B, *, rms_gate=5.0, best_align_lag=6, use_abs=True):
    """
    Best cosine between A and B over small integer lags (shift B relative to A).
    Channels included = union of channels with RMS>rms_gate in either EI.
    Returns: (best_cos, nch_used, best_lag, overlap_len)
    """
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    # Defensive shape alignment
    C = min(A.shape[0], B.shape[0])
    T = min(A.shape[1], B.shape[1])
    A = A[:C, :T]
    B = B[:C, :T]

    if use_abs:
        A0 = np.abs(A)
        B0 = np.abs(B)
    else:
        A0 = A
        B0 = B

    rmsA = np.sqrt(np.mean(A0**2, axis=1))
    rmsB = np.sqrt(np.mean(B0**2, axis=1))
    chans = np.flatnonzero((rmsA > float(rms_gate)) | (rmsB > float(rms_gate))).astype(int)
    if chans.size == 0:
        # fallback: at least the strongest trough channel of A
        chans = np.array([int(np.argmin(A.min(axis=1)))], dtype=int)

    best_cos = -np.inf
    best_lag = 0

    a = A0[chans].reshape(-1).astype(np.float64)
    na = float(np.linalg.norm(a)) + 1e-12

    for lag in range(-int(best_align_lag), int(best_align_lag) + 1):
        Bs = _zero_pad_shift(B0, lag)
        b = Bs[chans].reshape(-1).astype(np.float64)
        nb = float(np.linalg.norm(b)) + 1e-12
        cos = float(a.dot(b) / (na * nb))
        if cos > best_cos:
            best_cos = cos
            best_lag = int(lag)

    overlap_len = int(T - abs(best_lag))
    return float(best_cos), int(chans.size), int(best_lag), overlap_len


# --- knobs (match your usual EI-cos conventions) ---
RMS_GATE = 5.0
BEST_ALIGN_LAG = 6
USE_ABS = True

# --- inputs ---
if "leaves" not in globals():
    raise NameError("`leaves` not found. Expected list returned by recursive_split_and_visualize().")

ei_ks = final_ei_full
ref_ch_plot = int(np.argmin(ei_ks.min(axis=1)))

have_positions = ("ei_positions" in globals()) and (globals()["ei_positions"] is not None)
ei_positions_local = np.asarray(globals()["ei_positions"], dtype=np.float32) if have_positions else None

# --- plot ---
for i, leaf in enumerate(leaves):
    ei_leaf = leaf.get("ei", None)
    if ei_leaf is None:
        print(f"[skip] leaf {i}: missing 'ei'")
        continue
    ei_leaf = recenter_ei_to_ref_trough(np.asarray(ei_leaf, dtype=np.float32), center_index=40)

    cos, nch, lag, olen = _best_cosine_eis(
        ei_ks, ei_leaf,
        rms_gate=RMS_GATE,
        best_align_lag=BEST_ALIGN_LAG,
        use_abs=USE_ABS
    )
    ei_leaf_al = _zero_pad_shift(ei_leaf, lag)

    Nleaf = None
    if "idx_abs" in leaf and leaf["idx_abs"] is not None:
        try:
            Nleaf = int(np.asarray(leaf["idx_abs"]).size)
        except Exception:
            Nleaf = None

    depth = leaf.get("depth", None)
    note  = leaf.get("note", "")

    title_bits = [
        f"leaf {i}",
        f"KS={ks_name}",
        f"cos={cos:.4f}",
        f"lag={lag:+d}",
        f"nch={nch}",
        f"olen={olen}",
    ]
    if depth is not None:
        title_bits.append(f"depth={int(depth)}")
    if Nleaf is not None:
        title_bits.append(f"N={Nleaf}")
    if note:
        title_bits.append(str(note))

    if pew is not None and ei_positions_local is not None:
        fig, ax = plt.subplots(figsize=(20, 12))
        pew.plot_ei_waveforms(
            [ei_ks, ei_leaf_al],
            ei_positions_local,
            ref_channel=ref_ch_plot,
            colors=["black", "red"],
            scale=70.0, box_height=1.0, box_width=50.0,
            ax=ax
        )
        ax.set_title(" | ".join(title_bits))
        plt.tight_layout()
        plt.show()
    else:
        # fallback: just overlay the ref channel waveform
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(ei_ks[ref_ch_plot], color="black", lw=2, alpha=0.9, label="KS")
        ax.plot(ei_leaf_al[ref_ch_plot], color="red", lw=2, alpha=0.9, label="leaf")
        ax.set_title(" | ".join(title_bits) + f" | ref_ch={ref_ch_plot}")
        ax.grid(alpha=0.25)
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# === Per-spike confidence: leaf EI vs your chosen KS/original EI ===
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# YOU set this explicitly
# -----------------------------
ei_ks = final_ei_full   # <-- replace with your exact EI variable
ks_name = "ei_best_centered_ks"  # optional, just for printing

# -----------------------------
# helpers
# -----------------------------
try:
    from joint_utils import recenter_ei_to_ref_trough
except Exception:
    def recenter_ei_to_ref_trough(ei: np.ndarray, center_index: int = 40) -> np.ndarray:
        ei = np.asarray(ei, dtype=np.float32)
        C, T = ei.shape
        ref_ch = int(np.argmin(ei.min(axis=1)))
        trough_idx = int(np.argmin(ei[ref_ch]))
        shift = int(center_index - trough_idx)
        if shift == 0:
            return ei.copy()
        out = np.zeros_like(ei)
        if shift > 0:
            out[:, shift:] = ei[:, :T-shift]
        else:
            s = -shift
            out[:, :T-s] = ei[:, s:]
        return out

def _zero_pad_shift(ei: np.ndarray, lag: int) -> np.ndarray:
    ei = np.asarray(ei, dtype=np.float32)
    C, T = ei.shape
    lag = int(lag)
    if lag == 0:
        return ei
    out = np.zeros_like(ei)
    if lag > 0:
        out[:, lag:] = ei[:, :T-lag]
    else:
        s = -lag
        out[:, :T-s] = ei[:, s:]
    return out

def _robust_std(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return 1.0
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return float(1.4826 * mad + 1e-12)

def _best_sse_over_lags(S: np.ndarray, T: np.ndarray, lags: np.ndarray):
    """
    S: [N, D] snippets flattened
    T: [D] template flattened
    Returns:
      best_sse: [N]
      best_lag: [N] (lag that gave best SSE)
      best_a  : [N] (best scalar amplitude a)
    Uses SSE = ||S||^2 - (<S,T>^2 / ||T||^2) with zero-pad lag shifts.
    """
    S = np.asarray(S, dtype=np.float32)
    T0 = np.asarray(T, dtype=np.float32)

    N, D = S.shape
    s_norm = np.sum(S * S, axis=1).astype(np.float64)  # [N]

    # build shifted templates matrix [K, D]
    K = int(len(lags))
    Tlags = np.zeros((K, D), dtype=np.float32)
    for k, lag in enumerate(lags):
        # T0 corresponds to [M,L] flattened; shifting is along time within each channel.
        # We handle shift on [M,L] before flattening for correctness; so caller passes that.
        Tlags[k] = T0[k]  # placeholder; caller will pass pre-shifted rows

    # caller already provides a [K,D] in T0; if so, detect and use it
    if T0.ndim == 2 and T0.shape[0] == K and T0.shape[1] == D:
        Tlags = T0
    else:
        raise ValueError("Internal: expected precomputed Tlags [K,D].")

    t_norm = np.sum(Tlags * Tlags, axis=1).astype(np.float64) + 1e-12  # [K]
    dots = (S @ Tlags.T).astype(np.float64)  # [N,K]
    sse = s_norm[:, None] - (dots * dots) / t_norm[None, :]  # [N,K]

    kbest = np.argmin(sse, axis=1)
    best_sse = sse[np.arange(N), kbest]
    best_lag = np.asarray(lags, dtype=int)[kbest]
    best_a = dots[np.arange(N), kbest] / t_norm[kbest]
    return best_sse, best_lag, best_a

def per_spike_confidence_leaf_vs_ks(
    snips: np.ndarray,            # [C, L, Nall]
    idx_abs: np.ndarray,          # [Nleaf] absolute indices into snips
    ei_leaf: np.ndarray,          # [C, L]
    ei_ks: np.ndarray,            # [C, L]
    *,
    rms_gate: float = 5.0,
    best_align_lag: int = 6,
    use_abs: bool = True,
    chunk: int = 2000
):
    """
    Returns dict with per-spike fields:
      margin = SSE_KS - SSE_LEAF (positive means leaf fits better)
      conf   = sigmoid(margin / robust_std(margin))
      plus best lags and amplitude scales.
    """
    idx_abs = np.asarray(idx_abs, dtype=int).ravel()
    if idx_abs.size == 0:
        return None

    # shape guard + recenter both templates consistently
    C = min(ei_leaf.shape[0], ei_ks.shape[0], snips.shape[0])
    L = min(ei_leaf.shape[1], ei_ks.shape[1], snips.shape[1])
    ei_leaf = recenter_ei_to_ref_trough(np.asarray(ei_leaf[:C, :L], dtype=np.float32), center_index=40)
    ei_ks   = recenter_ei_to_ref_trough(np.asarray(ei_ks[:C, :L],   dtype=np.float32), center_index=40)

    # channel selection: union RMS gate on templates (optionally abs)
    A = np.abs(ei_leaf) if use_abs else ei_leaf
    B = np.abs(ei_ks)   if use_abs else ei_ks
    rmsA = np.sqrt(np.mean(A*A, axis=1))
    rmsB = np.sqrt(np.mean(B*B, axis=1))
    chans = np.flatnonzero((rmsA > rms_gate) | (rmsB > rms_gate)).astype(int)
    if chans.size == 0:
        chans = np.array([int(np.argmin(ei_ks.min(axis=1)))], dtype=int)

    lags = np.arange(-int(best_align_lag), int(best_align_lag) + 1, dtype=int)
    K = int(lags.size)

    # precompute shifted flattened templates [K, D]
    def build_Tlags(ei_template):
        # [M,L] -> shifted -> flatten
        M = chans.size
        D = M * L
        out = np.zeros((K, D), dtype=np.float32)
        base = ei_template[chans, :]
        for k, lag in enumerate(lags):
            out[k] = _zero_pad_shift(base, lag).reshape(-1)
        return out

    Tleaf = build_Tlags(np.abs(ei_leaf) if use_abs else ei_leaf)
    Tks   = build_Tlags(np.abs(ei_ks)   if use_abs else ei_ks)

    # allocate outputs
    N = idx_abs.size
    sse_leaf = np.empty(N, dtype=np.float64)
    sse_ks   = np.empty(N, dtype=np.float64)
    lag_leaf = np.empty(N, dtype=int)
    lag_ks   = np.empty(N, dtype=int)
    a_leaf   = np.empty(N, dtype=np.float64)
    a_ks     = np.empty(N, dtype=np.float64)

    # chunked compute to avoid RAM spikes
    for i0 in range(0, N, int(chunk)):
        i1 = min(N, i0 + int(chunk))
        idx = idx_abs[i0:i1]

        # snippets: [M,L,n] without advanced-index broadcasting issues
        S = np.asarray(snips[chans, :L, :], dtype=np.float32)    # [M,L,Nall]
        S = S[:, :, idx]                                         # [M,L,n]
        S = np.transpose(S, (2, 0, 1)).reshape(i1 - i0, -1)      # [n, D]
        if use_abs:
            S = np.abs(S)

        # best SSE over lags for each template
        sseL, lagL, aL = _best_sse_over_lags(S, Tleaf, lags)
        sseK, lagK, aK = _best_sse_over_lags(S, Tks,   lags)

        sse_leaf[i0:i1] = sseL
        sse_ks[i0:i1]   = sseK
        lag_leaf[i0:i1] = lagL
        lag_ks[i0:i1]   = lagK
        a_leaf[i0:i1]   = aL
        a_ks[i0:i1]     = aK

    margin = sse_ks - sse_leaf  # + means leaf fits better
    scale = _robust_std(margin)
    conf = 1.0 / (1.0 + np.exp(-margin / scale))

    return dict(
        idx_abs=idx_abs,
        chans=chans,
        margin=margin,
        conf=conf,
        sse_leaf=sse_leaf,
        sse_ks=sse_ks,
        lag_leaf=lag_leaf,
        lag_ks=lag_ks,
        a_leaf=a_leaf,
        a_ks=a_ks,
        scale=scale
    )

# -----------------------------
# run across all leaves
# -----------------------------
if "leaves" not in globals():
    raise NameError("`leaves` not found in globals.")

conf_leaves = []
ei_ks_use = np.asarray(ei_ks, dtype=np.float32)

print(f"[per-spike conf] KS template = {ks_name} | leaves={len(leaves)}")
for i, leaf in enumerate(leaves):
    idx = leaf.get("idx_abs", None)
    eiL = leaf.get("ei", None)
    if idx is None or eiL is None:
        print(f"  leaf {i}: missing idx_abs or ei, skip")
        continue

    out = per_spike_confidence_leaf_vs_ks(
        snips, idx, eiL, ei_ks_use,
        rms_gate=5.0,
        best_align_lag=6,
        use_abs=True,
        chunk=2000
    )
    conf_leaves.append(out)

    m = out["margin"]
    c = out["conf"]
    frac_leaf_better = float(np.mean(m > 0))
    frac_hi = float(np.mean(c > 0.9))
    frac_lo = float(np.mean(c < 0.6))
    print(f"  leaf {i}: N={m.size} | mean(conf)={c.mean():.3f} | leaf_better={frac_leaf_better:.3f} | conf>0.9={frac_hi:.3f} | conf<0.6={frac_lo:.3f}")

# optional quick look: combined confidence histogram
all_conf = np.concatenate([d["conf"] for d in conf_leaves if d is not None], axis=0)
plt.figure(figsize=(8,4))
plt.hist(all_conf, bins=50)
plt.title("Per-spike confidence (all leaves): sigmoid( (SSE_KS - SSE_leaf) / robust_scale )")
plt.xlabel("confidence")
plt.ylabel("count")
plt.tight_layout()
plt.show()